# Run Artificial Societies Survey

This notebook:
1. Creates 100 dummy personas in the format expected by the repo (`respondent_id`, `persona`)
2. Defines dummy survey questions in the expected dict format
3. Queries an LLM via OpenRouter and saves results as CSV

## Setup

In [ ]:
import os
import time
import pandas as pd
import requests
from datetime import datetime, timezone
from pathlib import Path
from tqdm.notebook import tqdm

In [ ]:
# --- CONFIGURATION ---
# Set your OpenRouter API key here or as an environment variable
API_KEY = os.getenv("OPENROUTER_API_KEY", "YOUR_KEY_HERE")
MODEL = "openai/gpt-4o-mini"  # choose any OpenRouter model
YEAR = 2024
RUNS = 1  # number of runs per persona-question pair

# Paths (relative to repo root)
PROJECT_DIR = Path("generation")
PERSONAS_FILE = PROJECT_DIR / "data" / "my_personas.csv"
OUTPUT_DIR = PROJECT_DIR / "synthetic_data" / f"year_{YEAR}"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OPENROUTER_API_URL = "https://openrouter.ai/api/v1/chat/completions"

## 1. Create 100 Personas

In [ ]:
import random
random.seed(42)

# Demographic building blocks
ages = list(range(18, 80))
sexes = ["male", "female"]
races = ["White", "Black or African American", "Hispanic", "Asian"]
marital = ["never been married", "married", "divorced", "widowed"]
education = [
    "completed high school",
    "completed some college",
    "completed 4 years of college (bachelor's degree)",
    "completed a graduate degree",
]
income = [
    "$15,000 to $24,999", "$25,000 to $39,999", "$40,000 to $59,999",
    "$60,000 to $74,999", "$75,000 to $109,999", "$110,000 to $129,999",
]
party = [
    "a strong Democrat", "a not very strong Democrat", "an independent",
    "a not very strong Republican", "a strong Republican",
]
ideology = [
    "extremely liberal", "liberal", "slightly liberal",
    "moderate (middle of the road)",
    "slightly conservative", "conservative", "extremely conservative",
]
religion = [
    "never attend religious services",
    "attend religious services several times a year",
    "attend religious services every week",
]

personas = []
for i in range(1, 101):
    text = (
        f"I am {random.choice(ages)} years old. "
        f"I am {random.choice(sexes)}. "
        f"I am {random.choice(races)}. "
        f"I have {random.choice(marital)}. "
        f"I {random.choice(education)}. "
        f"My family income is {random.choice(income)}. "
        f"I am {random.choice(party)}. "
        f"I am {random.choice(ideology)}. "
        f"I {random.choice(religion)}."
    )
    personas.append({"respondent_id": i, "persona": text})

personas_df = pd.DataFrame(personas)
personas_df.to_csv(PERSONAS_FILE, index=False)
print(f"Created {len(personas_df)} personas -> {PERSONAS_FILE}")
personas_df.head(3)

## 2. Define Dummy Questions

Same dict format as the repo: `{variable_name: {"text": ..., "options": {int: str}}}`

In [ ]:
QUESTIONS = {
    "dummy_abortion": {
        "text": "Please tell me whether or not you think it should be possible for a pregnant woman to obtain a legal abortion if the woman's own health is seriously endangered by the pregnancy?",
        "options": {1: "Yes", 2: "No"},
    },
    "dummy_gunlaw": {
        "text": "Would you favor or oppose a law which would require a person to obtain a police permit before he or she could buy a gun?",
        "options": {1: "Favor", 2: "Oppose"},
    },
    "dummy_trust": {
        "text": "Generally speaking, would you say that most people can be trusted or that you can't be too careful in dealing with people?",
        "options": {1: "Can trust", 2: "Can't be too careful", 3: "Depends"},
    },
}

print(f"Defined {len(QUESTIONS)} questions: {list(QUESTIONS.keys())}")

## 3. Query LLM via OpenRouter

In [ ]:
def query_openrouter(model, persona, question, options, api_key, year, max_retries=3):
    """Query OpenRouter for a single persona-question pair. Returns dict with answer and metadata."""
    options_text = "\n".join([f"{k}. {v}" for k, v in options.items()])

    prompt = (
        f"It is now {year}. You are answering survey questions as the following person, "
        f"who is living in the United States:\n\n{persona}\n\n"
        f"Question: {question}\n\nOptions:\n{options_text}\n\n"
        f'Respond with ONLY the number of your answer (e.g., "1" or "2"). Do not explain your reasoning.'
    )

    headers = {"Authorization": f"Bearer {api_key}", "Content-Type": "application/json"}
    body = {
        "model": model,
        "messages": [{"role": "user", "content": prompt}],
        "temperature": 1.0,
        "max_tokens": 50,
    }

    for attempt in range(max_retries):
        try:
            resp = requests.post(OPENROUTER_API_URL, headers=headers, json=body, timeout=30)
            resp.raise_for_status()
            result = resp.json()
            answer_text = result["choices"][0]["message"]["content"].strip()
            usage = result.get("usage", {})

            try:
                answer = int(answer_text)
                if answer not in options:
                    return {"answer": None, "error": f"Invalid: {answer_text}",
                            "prompt_tokens": usage.get("prompt_tokens", 0),
                            "completion_tokens": usage.get("completion_tokens", 0),
                            "raw_response": answer_text}
            except ValueError:
                return {"answer": None, "error": f"Parse error: {answer_text}",
                        "prompt_tokens": usage.get("prompt_tokens", 0),
                        "completion_tokens": usage.get("completion_tokens", 0),
                        "raw_response": answer_text}

            return {"answer": answer, "error": None,
                    "prompt_tokens": usage.get("prompt_tokens", 0),
                    "completion_tokens": usage.get("completion_tokens", 0),
                    "raw_response": answer_text}

        except Exception as e:
            if attempt < max_retries - 1:
                time.sleep(2 ** attempt)
            else:
                return {"answer": None, "error": str(e),
                        "prompt_tokens": 0, "completion_tokens": 0, "raw_response": ""}

In [ ]:
# Build task list
tasks = []
for _, row in personas_df.iterrows():
    for var_name, q in QUESTIONS.items():
        for run in range(1, RUNS + 1):
            tasks.append({
                "persona_id": row["respondent_id"],
                "persona_text": row["persona"],
                "variable": var_name,
                "question": q["text"],
                "options": q["options"],
                "run": run,
            })

print(f"Total API calls: {len(tasks)} ({len(personas_df)} personas x {len(QUESTIONS)} questions x {RUNS} runs)")

In [ ]:
# Run queries and collect results
model_filename = MODEL.replace("/", "_")
output_file = OUTPUT_DIR / f"{model_filename}.csv"

results = []
for task in tqdm(tasks, desc=MODEL.split("/")[-1]):
    r = query_openrouter(
        MODEL, task["persona_text"], task["question"], task["options"], API_KEY, YEAR
    )
    results.append({
        "timestamp": datetime.now(timezone.utc).isoformat(),
        "model": MODEL,
        "persona_id": task["persona_id"],
        "variable": task["variable"],
        "question_short": task["question"][:50] + "...",
        "run": task["run"],
        "answer": r.get("answer"),
        "prompt_tokens": r.get("prompt_tokens", 0),
        "completion_tokens": r.get("completion_tokens", 0),
        "total_tokens": r.get("prompt_tokens", 0) + r.get("completion_tokens", 0),
        "error": r.get("error", ""),
        "raw_response": r.get("raw_response", ""),
    })

results_df = pd.DataFrame(results)
results_df.to_csv(output_file, index=False)
print(f"\nSaved {len(results_df)} results -> {output_file}")

In [ ]:
# Quick summary
success_rate = (results_df["answer"].notna().sum() / len(results_df)) * 100
print(f"Success rate: {success_rate:.1f}%")
print(f"Total tokens: {results_df['total_tokens'].sum():,}")
print(f"\nAnswer distribution per question:")
for var in QUESTIONS:
    subset = results_df[results_df["variable"] == var]
    print(f"  {var}: {subset['answer'].value_counts().sort_index().to_dict()}")